In [1]:
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# 定义大模型
import os
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    extra_body={"thinking":{"type":"disabled"}}# 关闭思考输出
)

In [3]:
# 定义工具，使用tavily来进行联网搜索
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
tavily=TavilySearch(
    maxresults=5,
    topic="general"
)
@tool
def web_search(query:str) -> str:
    """Search the web for information"""
    return tavily.invoke(query)


In [4]:
# 记忆 使用Sqlite作为存储方案
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
# 连接sqlite
connection=sqlite3.connect("resources/chef.db",check_same_thread=False)
# 初始化checkpoint
checkpoint = SqliteSaver(connection)
# 自动建表
checkpoint.setup()

In [5]:
# 定义智能体
from langchain.agents import create_agent
system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由，帮助用户快速做出决策。
5.请从公开且无防盗链限制的图片源（如Unsplash、Pexels等开放CDN）查找图片，且保证图片URL可直接在任意网页中嵌入显示，不验证Referer。
请严格按照流程，优先调用 web_search 工具搜索食谱，再搜索不到的情况下才能自己发挥。
"""
agent=create_agent(
    model=model,
    checkpointer=checkpoint,
    tools=[web_search],
    system_prompt=system_prompt
)

## 本地图片数据上传

In [18]:
# 创建上传组件，用于模拟图片上传
from ipywidgets import FileUpload
from IPython.display import display
uploader = FileUpload(accept="*",multiple=False)
display(uploader)

FileUpload(value=(), accept='*', description='Upload')

In [19]:
print(uploader.value)

()


In [20]:
# 读取图片，转为base64字符串
import base64
# 获取第一个，也是唯一一个上传的文件
uploaded_file = uploader.value[0]
# 获取其内存视图
content_mv = uploaded_file["content"]
# 转换内存视图 -> 字节
img_bytes = bytes(content_mv)
# base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

IndexError: tuple index out of range

In [15]:
# 组织多模态消息
from langchain_core.messages import HumanMessage
multimodal_question = HumanMessage(content=[
    {
        "type":"image",
        "base64":img_b64,
        "mime_type":"image/jpeg"
    },
    {"type":"text","text":"帮我看看这些食材能做些什么？根据图片里的食材给我推荐几道菜"}
])

NameError: name 'img_b64' is not defined

## 测试

In [9]:
# 设定thread_id,作为会话标识
config = {"configurable":{"thread_id":"thread_1"}}

流式调用

In [ ]:
for chunk,metadata in agent.stream(
    {"messages":[multimodal_question]},
    stream_mode = "messages",
    config = config# 调用时添加thread_id
):
    print(chunk.content,end="",flush=True)

阻塞式调用

In [12]:
response=agent.invoke({"messages":[multimodal_question]},config)

In [14]:
# 友好打印
for message in response['messages']:
    message.pretty_print()

# 报错原因：在3秒的时间窗口内，Jupyter服务器向客户端发送的输出数据总量超过了1,000,000字节（约 1 MB）的速率限制。通常是因为代码循环打印大量内容、流式输出过多 token、或输出了超长数据结构（如完整的大型 DataFrame 或嵌套字典）导致。

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [15]:
# 只输出结果和消耗token
print(response["messages"][-1].content)
token=response["messages"][-1].usage_metadata["total_tokens"]
print(f"token: {token}")

好的，搜索结果显示了多种利用这些食材的绝佳方案。现在我将对它们进行评估和排序。

**食谱评估与排序:**

1.  **腊肠时蔬焖饭**
    *   **匹配度**: 极高。此食谱能完美利用您提供的所有六种食材。
    *   **制作难度**: ⭐ (非常简单)。所有食材切丁后与米饭一同放入电饭煲即可，是典型的“懒人料理”。
    *   **营养价值**: ⭐⭐⭐⭐⭐ (非常均衡)。一锅端包含了主食（米饭）、优质蛋白（腊肠）和多种维生素、膳食纤维（土豆、胡萝卜、玉米、青豆、香菇），营养全面。
    *   **综合得分**: 9.8/10

2.  **腊肠小炒 (玉米青豆炒腊肠)**
    *   **匹配度**: 高。主要使用腊肠、玉米和青豆，也可根据喜好加入胡萝卜丁或香菇片。
    *   **制作难度**: ⭐⭐ (简单)。需要基本的翻炒技巧，烹饪时间短。
    *   **营养价值**: ⭐⭐⭐⭐ (良好)。蛋白质和蔬菜搭配合理，色彩鲜艳，能促进食欲。如果作为主菜，建议额外搭配一份主食。
    *   **综合得分**: 8.5/10

3.  **香菇土豆炖腊肠**
    *   **匹配度**: 中高。以腊肠、香菇、土豆为核心，胡萝卜可作为配菜增加风味和营养。
    *   **制作难度**: ⭐⭐ (简单)。属于家常炖菜，步骤不复杂。
    *   **营养价值**: ⭐⭐⭐⭐ (良好)。腊肠的咸香与香菇的鲜美、土豆的软糯相结合，是一道非常下饭的菜肴。同样建议搭配主食食用。
    *   **综合得分**: 8.0/10

---

### 4. 结构化方案输出

根据您的食材，我为您整理了以下三份烹饪建议报告，您可以根据自己的时间和口味偏好进行选择。

#### **首选推荐：一锅出的美味 - 腊肠时蔬焖饭**

*   **推荐理由**: 这是最能发挥您所有食材优势的菜谱。做法极其简单，只需一个电饭煲，就能得到一顿营养均衡、色香味俱全的完美餐食。腊肠的油脂和香气会渗透到每一粒米饭和每一块蔬菜中，味道绝佳。
*   **制作难度**: ⭐ (非常简单)
*   **营养指数**: ⭐⭐⭐⭐⭐ (满分)
*   **简要步骤**:
    1.  将腊肠、土豆、胡萝卜、香菇全部切成小丁。
    2.  淘洗好大米，放

In [17]:
response=agent.invoke({"messages":[HumanMessage(content="我喜欢第三道菜，可以说的详细点吗")]},config)
print(response["messages"][-1].content)

太棒了！您的眼光真好，**香菇土豆炖腊肠**是一道非常经典的“米饭杀手”菜肴。腊肠的咸香油脂渗入软糯的土豆和鲜美的香菇中，汤汁浓郁，口感丰富。

既然您选择了这道菜，我为您准备了一份**详细的私人厨师版烹饪指南**，包含备菜技巧、调味秘籍和让味道升级的小贴士。

---

### 🍲 菜谱名称：浓香香菇土豆炖腊肠

#### **🛒 食材准备（基于您现有的食材）**
*   **主料**：
    *   腊肠：2根
    *   土豆：1个（大）
    *   香菇：3朵
    *   胡萝卜：半根（作为配色和增加甜味，强烈建议加入）
*   **辅料**（家中常备）：
    *   食用油：少许
    *   生姜：2-3片（去腥增香，可选）
    *   大蒜：2瓣（拍碎，可选）
    *   热水：适量
*   **调味料**：
    *   生抽（酱油）：2汤匙
    *   老抽（深色酱油）：半汤匙（主要为了上色，让菜品红亮诱人）
    *   蚝油：1汤匙（提鲜关键）
    *   白糖：1茶匙（中和咸味，提鲜）
    *   白胡椒粉：少许（可选，增加风味层次）
    *   盐：视情况少量添加（因为腊肠、生抽、蚝油都有咸味）

---

#### **🔪 第一步：精细备菜（成功的关键）**

1.  **处理腊肠**：
    *   将腊肠斜切成**薄片**。斜切可以增加受热面积，更容易炒出油脂，口感也更好。
    *   *小贴士*：如果腊肠比较硬，可以先上锅蒸5分钟变软后再切，或者在温水中泡一下。

2.  **处理土豆与胡萝卜**：
    *   土豆去皮，切成**滚刀块**（即切一刀转一下土豆，切成不规则的多面体）。滚刀块比方块更容易入味，且炖煮时不易散烂。
    *   胡萝卜同样去皮，切成稍大的滚刀块或厚片。
    *   *重要*：切好的土豆如果不马上用，请泡在冷水里防止氧化变黑。

3.  **处理香菇**：
    *   香菇洗净，去掉根部硬的蒂。
    *   如果香菇较大，可以对半切开或切成厚片；如果较小，保持完整或对半切即可，这样口感更饱满。

4.  **辅料准备**：
    *   姜切片，蒜拍碎备用。

---

#### **🍳 第二步：烹饪步骤**

**1. 煸炒腊肠（出

In [11]:
checkpoint.delete_thread(thread_id="thread_1")# 删除对话历史

# 获取对话历史

In [10]:
# 根据config获取对应的对话历史
checkpoint.get(config)

{'v': 4,
 'ts': '2026-07-29T12:52:34.097994+00:00',
 'id': '1f18b4c5-e4b8-6965-800b-bf59de03d38c',
 'channel_values': {'messages': [HumanMessage(content=[{'type': 'image', 'base64': '...', 'mime_type': 'image/jpeg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？根据图片里的食材给我推荐几道菜'}], additional_kwargs={}, response_metadata={}, id='63d52767-20eb-40f0-9798-e94f56b18160'),
   AIMessage(content='好的，收到您的食材照片。作为一名私人厨师，我将立即为您分析这些食材并规划美味的菜肴。\n\n### 1. 识别和评估食材\n\n根据您提供的图片，我已识别出以下新鲜且优质的食材：\n\n*   **腊肠**: 2根，色泽红润，肥瘦相间，看起来品质很好，是提供风味和蛋白质的关键。\n*   **土豆**: 1个，个大饱满，表皮光滑，适合炖煮或蒸制。\n*   **香菇**: 3朵，菇盖厚实，香气浓郁，能极大提升菜肴的鲜味。\n*   **胡萝卜**: 半根，颜色鲜艳，质地坚实，富含营养且能增加甜味。\n*   **玉米粒**: 一小碗，颗粒金黄饱满，无论是新鲜的还是冷冻的，都能为菜品增添色彩和清甜口感。\n*   **青豆**: 一小勺，颜色翠绿，看起来非常新鲜，能提供清新的口感和丰富的维生素。\n\n**当前可用食材清单:**\n*   腊肠\n*   土豆\n*   香菇\n*   胡萝卜\n*   玉米粒\n*   青豆\n\n---\n\n### 2. & 3. 智能食谱检索与多维度评估\n\n接下来，我将以这份“可用食材清单”为核心，为您搜索并筛选出最合适的菜谱。我会从**制作难度**和**营养价值**两个维度进行量化评分（满分10分），并将简单易做且营养均衡的菜谱优先推荐给您。\n\n', additional_kwargs={'refusal': None}, response_metadata

In [11]:
def get_messages(thread_id: str) -> list[dict[str, str]]:
    """获取会话历史"""

    # 根据 thread_id 查询 checkpoint
    cp = checkpoint.get({"configurable": {"thread_id": thread_id}})

    # 如果不存在，返回空列表
    if not cp:
        return []

    # 安全获取 messages
    channel_values = cp.get("channel_values")
    if not channel_values:
        return []

    messages = channel_values.get("messages", [])
    if not messages:
        return []

    # 转换消息格式
    result = []
    for msg in messages:
        if not msg.content:
            continue

        if isinstance(msg, HumanMessage):
            result.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            result.append({"role": "assistant", "content": msg.content})

    return result

In [26]:
# 根据线程id获取对话历史
from langchain_core.messages import HumanMessage,AIMessage# 只看对话历史不再次运行的话要添加导入
get_messages("thread_1")

[]